# 🔥 Notebook 6: Hot Keys and Advanced Patterns

When one key receives disproportionate traffic, it becomes a bottleneck. Learn to detect and handle hot keys.

## Learning Objectives

By the end of this notebook, you'll understand:
- What causes hot keys
- Detection strategies
- Fixed-k key splitting
- Dynamic key splitting
- Real-world examples

In [ ]:
import redis
import random
import time
from collections import defaultdict
from typing import Dict, List, Optional

r = redis.Redis(host='localhost', port=6379, decode_responses=True)
r.flushall()

print("✅ Connected to Redis!")
print("📊 Open RedisInsight: http://localhost:5540")

## 🔥 The Hot Key Problem

In [ ]:
print("🔥 The Hot Key Problem")
print("=" * 60)
print("""
SCENARIO: Celebrity tweet during Super Bowl
─────────────────────────────────────────────────────────────

Taylor Swift tweets: "Go Chiefs! 🏈"

┌────────────────────────────────────────────────────────────┐
│    1 million likes in 60 seconds                          │
│                                                            │
│    All 1M writes go to: tweet_id = 12345                  │
│                                                            │
│    Shard 3 (where 12345 lives): 💥 OVERLOADED!            │
│    Other shards: 😴 Idle                                  │
└────────────────────────────────────────────────────────────┘

PROBLEM:
• Sharding doesn't help - all traffic goes to ONE shard
• That shard becomes bottleneck
• Other shards sit idle

HOT KEY CAUSES:
• Celebrity posts (millions of followers)
• Flash sales (one product_id)
• Breaking news (one article_id)
• Live events (one event_id)
""")

In [ ]:
print("🔬 Simulating Hot Key Problem")
print("=" * 60)

shard_counts = defaultdict(int)
NUM_SHARDS = 10

def get_shard(key: int) -> int:
    return key % NUM_SHARDS

regular_posts = list(range(1, 10001))
celebrity_post = 12345

print("\n📝 Normal traffic: 10,000 likes across 10,000 posts")
for post_id in regular_posts:
    shard_counts[get_shard(post_id)] += 1

print("📊 Shard distribution (normal):")
for shard in range(NUM_SHARDS):
    bar = "█" * (shard_counts[shard] // 100)
    print(f"   Shard {shard}: {shard_counts[shard]:>5} writes {bar}")

shard_counts.clear()

print("\n🔥 Hot key traffic: 10,000 likes on ONE celebrity post")
for _ in range(10000):
    shard_counts[get_shard(celebrity_post)] += 1

print("📊 Shard distribution (hot key):")
for shard in range(NUM_SHARDS):
    bar = "█" * (shard_counts[shard] // 100)
    print(f"   Shard {shard}: {shard_counts[shard]:>5} writes {bar}")

print("\n❌ All traffic goes to shard 5 (12345 % 10 = 5)!")

## 🔍 Hot Key Detection

In [ ]:
class HotKeyDetector:
    def __init__(self, window_size: int = 100, threshold_ratio: float = 0.1):
        self.window_size = window_size
        self.threshold_ratio = threshold_ratio
        self.counts: Dict[str, int] = defaultdict(int)
        self.total_writes = 0
        self.hot_keys: set = set()
    
    def record(self, key: str):
        self.counts[key] += 1
        self.total_writes += 1
        
        if self.total_writes % self.window_size == 0:
            self._detect_hot_keys()
    
    def _detect_hot_keys(self):
        threshold = self.total_writes * self.threshold_ratio
        for key, count in self.counts.items():
            if count > threshold:
                if key not in self.hot_keys:
                    self.hot_keys.add(key)
                    print(f"   🔥 HOT KEY DETECTED: {key} ({count} writes, {count/self.total_writes*100:.1f}%)")
    
    def is_hot(self, key: str) -> bool:
        return key in self.hot_keys

print("🔍 Hot Key Detection Demo")
print("=" * 60)

detector = HotKeyDetector(window_size=100, threshold_ratio=0.1)

print("\n📝 Writing traffic (mostly normal, some hot)...")
for i in range(1000):
    if random.random() < 0.3:
        detector.record("celebrity_post_123")
    else:
        detector.record(f"normal_post_{random.randint(1, 100)}")

print(f"\n📊 Summary:")
print(f"   Total writes: {detector.total_writes}")
print(f"   Hot keys found: {detector.hot_keys}")

## 🔀 Fixed-K Key Splitting

In [ ]:
print("🔀 Fixed-K Key Splitting")
print("=" * 60)
print("""
IDEA: Split hot key into K sub-keys, spread across shards

BEFORE:
─────────────────────────────────────────────────────────────
    tweet_12345 → Shard 5 (ALL writes here!)

AFTER (K=4):
─────────────────────────────────────────────────────────────
    tweet_12345_0 → Shard 2
    tweet_12345_1 → Shard 7
    tweet_12345_2 → Shard 1  
    tweet_12345_3 → Shard 9

WRITES:
    Writer picks random sub-key: tweet_12345_{random(0,3)}
    Spreads load across 4 shards!

READS:
    Total likes = sum(tweet_12345_0 ... tweet_12345_3)
    Query all K sub-keys and sum
""")

In [ ]:
class FixedKSplitter:
    def __init__(self, k: int = 4):
        self.k = k
        self.hot_keys: set = set()
    
    def mark_hot(self, key: str):
        self.hot_keys.add(key)
    
    def get_write_key(self, key: str) -> str:
        if key in self.hot_keys:
            suffix = random.randint(0, self.k - 1)
            return f"{key}_{suffix}"
        return key
    
    def get_read_keys(self, key: str) -> List[str]:
        if key in self.hot_keys:
            return [f"{key}_{i}" for i in range(self.k)]
        return [key]
    
    def increment(self, key: str):
        write_key = self.get_write_key(key)
        r.incr(write_key)
        return write_key
    
    def get_total(self, key: str) -> int:
        read_keys = self.get_read_keys(key)
        total = 0
        for rk in read_keys:
            val = r.get(rk)
            if val:
                total += int(val)
        return total

print("🔬 Fixed-K Splitting Demo")
print("=" * 60)

splitter = FixedKSplitter(k=4)
splitter.mark_hot("celebrity_tweet")

print("\n📝 Writing 1000 likes to hot key (split into 4)...")
write_distribution = defaultdict(int)
for _ in range(1000):
    written_key = splitter.increment("celebrity_tweet")
    write_distribution[written_key] += 1

print("\n📊 Write Distribution:")
for key in sorted(write_distribution.keys()):
    count = write_distribution[key]
    bar = "█" * (count // 25)
    print(f"   {key}: {count:>4} writes {bar}")

print(f"\n📖 Reading total:")
total = splitter.get_total("celebrity_tweet")
print(f"   Total likes: {total}")

print("\n✅ Hot key spread across 4 sub-keys!")

## 📈 Dynamic Key Splitting

In [ ]:
print("📈 Dynamic Key Splitting")
print("=" * 60)
print("""
PROBLEM: Fixed K is wasteful for varying traffic levels
─────────────────────────────────────────────────────────────

• Low traffic: K=4 means 4 reads for every query (overkill)
• High traffic: K=4 might not be enough

SOLUTION: Dynamically adjust K based on traffic
─────────────────────────────────────────────────────────────

Strategy:
1. Start with K=1 (no splitting)
2. Monitor write rate per key
3. If rate > threshold, double K
4. Migrate data to new sub-keys

Example:
    Normal: celebrity_tweet (K=1)
    Hot:    celebrity_tweet_0, celebrity_tweet_1 (K=2)
    Viral:  celebrity_tweet_0..3 (K=4)
""")

In [ ]:
class DynamicSplitter:
    def __init__(self, rate_threshold: int = 100):
        self.rate_threshold = rate_threshold
        self.key_k: Dict[str, int] = defaultdict(lambda: 1)
        self.write_counts: Dict[str, int] = defaultdict(int)
        self.check_interval = 50
    
    def increment(self, key: str):
        self.write_counts[key] += 1
        
        if self.write_counts[key] % self.check_interval == 0:
            self._check_and_split(key)
        
        k = self.key_k[key]
        suffix = random.randint(0, k - 1)
        write_key = f"{key}_{suffix}" if k > 1 else key
        r.incr(write_key)
        return k
    
    def _check_and_split(self, key: str):
        current_k = self.key_k[key]
        writes_per_subkey = self.write_counts[key] / current_k
        
        if writes_per_subkey > self.rate_threshold:
            new_k = current_k * 2
            self.key_k[key] = new_k
            print(f"   📈 Scaling {key}: K={current_k} → K={new_k}")
    
    def get_total(self, key: str) -> int:
        k = self.key_k[key]
        if k == 1:
            val = r.get(key)
            return int(val) if val else 0
        
        total = 0
        for i in range(k):
            val = r.get(f"{key}_{i}")
            if val:
                total += int(val)
        return total

print("🔬 Dynamic Splitting Demo")
print("=" * 60)

r.flushall()
dynamic = DynamicSplitter(rate_threshold=100)

print("\n📝 Simulating viral post (traffic increases over time)...")
print("\n   Phase 1: Normal traffic (100 writes)")
for _ in range(100):
    dynamic.increment("viral_post")
print(f"   Current K: {dynamic.key_k['viral_post']}")

print("\n   Phase 2: Going viral (400 more writes)")
for _ in range(400):
    dynamic.increment("viral_post")
print(f"   Current K: {dynamic.key_k['viral_post']}")

print("\n   Phase 3: Super viral (1000 more writes)")
for _ in range(1000):
    dynamic.increment("viral_post")
print(f"   Final K: {dynamic.key_k['viral_post']}")

print(f"\n📊 Total writes recorded: {dynamic.get_total('viral_post')}")
print("\n✅ Key automatically scaled from K=1 to K=8!")

## 🧪 Quick Quiz

1. **Why doesn't sharding help with hot keys?**

2. **What's the read trade-off of key splitting?**

3. **When would you prefer fixed-K vs dynamic splitting?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why sharding doesn't help:")
print("   - Hot key always maps to SAME shard")
print("   - All traffic concentrates on one node")
print("   - Other shards sit idle")
print()
print("2. Read trade-off of splitting:")
print("   - Must query K sub-keys instead of 1")
print("   - More network calls or parallel queries")
print("   - Sum results = higher read latency")
print()
print("3. Fixed-K vs Dynamic:")
print("   Fixed-K: Simpler, predictable reads")
print("           Good when hot keys are known")
print("   Dynamic: Adapts to changing traffic")
print("           Better for unpredictable spikes")

## 📚 Summary

### Key Takeaways

1. **Hot keys bypass sharding** - All traffic to one shard
2. **Detect early** - Monitor write rates per key
3. **Fixed-K splitting** - Simple, predictable, works for known hot keys
4. **Dynamic splitting** - Adapts to traffic, handles surprises
5. **Trade-off** - More writes per key = more reads to sum

### 🎉 Pattern Complete!

You've learned the core strategies for scaling writes:
1. **Database optimization** - Bulk inserts, minimal indexes
2. **Sharding** - Distribute across servers
3. **Queues** - Buffer bursts, steady drain
4. **Batching** - Combine writes, aggregate counters
5. **Hot keys** - Split to spread load

### Decision Framework

```
High write volume?
├── Single key hot? → Key splitting
├── Bursty traffic? → Queues + load shedding
├── Many small writes? → Batching + aggregation
└── Uniform load? → Sharding
```